# 第17章 Hugging Faceと事前学習済みモデル

世界中のモデルを借りる前に知っておくこと：ベースとチャット、ライセンス、プロンプト

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## ベースモデルとチャットモデル


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

gpu = torch.cuda.is_available()
dtype = torch.float16 if gpu else torch.float32

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, device_map="auto")
    return tok, model

base_tok, base = load("Qwen/Qwen2.5-0.5B")
chat_tok, chat = load("Qwen/Qwen2.5-0.5B-Instruct")
print("読み込み完了")

In [ ]:
def complete(tok, model, text, n=60):
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(complete(base_tok, base, "日本で一番高い山は何ですか？"))

In [ ]:
print(complete(base_tok, base, "問：日本で一番高い山は何ですか？\n答：", n=30))

In [ ]:
def ask(tok, model, messages, n=100):
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask(chat_tok, chat, [{"role": "user", "content": "日本で一番高い山は何ですか？"}]))

## チャットテンプレートと複数ターンの会話


In [ ]:
messages = [
    {"role": "system", "content": "あなたは親切なアシスタントです。"},
    {"role": "user", "content": "私の名前はミナトです。"},
    {"role": "assistant", "content": "こんにちは、ミナトさん！"},
    {"role": "user", "content": "私の名前は何でしたっけ？"},
]
print(chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

In [ ]:
print(ask(chat_tok, chat, messages))

## プロンプトの工夫と、その限界


In [ ]:
reviews = [
    ("このラーメン、スープが最高においしかった！", "肯定"),
    ("店員さんの態度が悪くて二度と行かない。", "否定"),
    ("値段のわりに量が少なくてがっかり。", "否定"),
    ("駅から近くて便利、また来たいです。", "肯定"),
    ("注文してから40分も待たされた。", "否定"),
    ("期待していなかったけど、意外とよかった。", "肯定"),
]

def classify(review, few_shot=False):
    msgs = [{"role": "system", "content": "レビューが肯定的なら「肯定」、否定的なら「否定」とだけ答えてください。"}]
    if few_shot:   # 例を2つ見せる
        msgs += [{"role": "user", "content": "最高の一日でした。"}, {"role": "assistant", "content": "肯定"},
                 {"role": "user", "content": "最悪の対応でした。"}, {"role": "assistant", "content": "否定"}]
    msgs.append({"role": "user", "content": review})
    return ask(chat_tok, chat, msgs, n=5).strip()

for few in [False, True]:
    correct = sum(classify(r, few) == label for r, label in reviews)
    print(("few-shot " if few else "zero-shot"), f"正解 {correct}/{len(reviews)}")

## 生成の設定を確かめる


In [ ]:
print(chat.generation_config)

## モデルの大きさとGPUメモリ


In [ ]:
def infer_gb(n_params_billion, bytes_per_param):
    return n_params_billion * bytes_per_param

print("        float16   8ビット   4ビット")
for size in [0.5, 1.5, 3, 7, 14]:
    print(f"{size:4}B  {infer_gb(size, 2):6.1f}GB {infer_gb(size, 1):6.1f}GB {infer_gb(size, 0.5):6.1f}GB")

## Hugging Faceのアカウントとトークン


In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))